# 08 - Automated Content Analytics Reporting Pipeline


## Digikala Content Analytics Project


## Business Problem

Content teams need continuous monitoring of:

- Product page quality
- Customer satisfaction
- Category performance
- Content improvement opportunities


Manual reporting is slow and difficult to scale.


## Objective

Build an automated analytics pipeline that:

- Calculates content KPIs
- Detects quality issues
- Generates actionable recommendations
- Supports data-driven decisions


## Business Impact

This system helps:

- Reduce manual analysis effort
- Improve content operations
- Enable scalable monitoring


In [1]:
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt


import warnings

warnings.filterwarnings("ignore")


print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
data = pd.read_csv(
    "../data/processed/content_analytics_dataset.csv"
)


data.head()

,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,...,Price,Seller,Is_Fake,min_price_last_month,sub_category,Content_Completeness,Recommendation_Score,Customer_Satisfaction,Engagement_Score,Opportunity_Score
0,53672599,پیشنهاد نمیشود,به درد نمیخوره,23 شهریور 1402,1.0,not_recommended,True,252058,NaN,NaN,...,199000,دیجی‌کالا,False,0,book & stationary & art,1.0,1,0.6,0.000000,0.0
1,9897229,بسته بندی بد,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,16 تیر 1399,0.0,recommended,True,252058,['تجربه جالبی بود برام '],['بسته بندی جالبی نداشت'],...,199000,دیجی‌کالا,False,0,book & stationary & art,1.0,1,0.5,0.003717,0.0
2,38074516,برس ریمل,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خ...,26 مرداد 1401,0.0,recommended,True,3331597,NaN,NaN,...,79000,پینک لندشاپ,False,0,beauty,1.0,1,0.5,0.000000,0.0
3,18628562,خوبه و خوشرنگ,به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد ...,28 اسفند 1399,0.0,recommended,True,3331329,NaN,NaN,...,80000,یانگوم,False,0,beauty,1.0,1,0.5,0.000000,0.0
4,53301258,برس رنگ مو,معمولیه اگه واسه خونه رنگ کردن شخصی میخواین او...,12 شهریور 1402,3.0,recommended,True,3255700,NaN,NaN,...,84100,گالری آرایشی به سیما,False,84100,beauty,1.0,1,0.8,0.000000,0.0


In [3]:
validation_report = pd.DataFrame({

    "Missing Values":
    data.isnull().sum(),

    "Unique Values":
    data.nunique(),

    "Data Type":
    data.dtypes

})


validation_report.head(20)

,Missing Values,Unique Values,Data Type
id,0,199993,int64
title,120457,39224,object
body,16,154038,object
created_at,0,2074,object
rate,0,114,float64
recommendation_status,35519,3,object
is_buyer,0,2,bool
product_id,0,16344,int64
advantages,221110,16304,object
disadvantages,232880,7229,object


In [4]:
def calculate_content_kpis(df):

    kpis = {

        "Total Products":
        df["product_id"].nunique(),


        "Average Rating":
        round(
            df["rate"].mean(),
            2
        ),


        "Average Content Quality":
        round(
            df["Content_Completeness"].mean(),
            2
        ),


        "Average Satisfaction":
        round(
            df["Customer_Satisfaction"].mean(),
            2
        )

    }


    return pd.DataFrame(
        kpis.items(),
        columns=[
            "Metric",
            "Value"
        ]
    )

In [5]:
kpi_report = calculate_content_kpis(data)

kpi_report

,Metric,Value
0,Total Products,16344.00
1,Average Rating,3.70
2,Average Content Quality,1.00
3,Average Satisfaction,0.75


In [6]:
category_monitoring = (

data

.groupby("Category1")

.agg(

products=
("product_id","nunique"),


content_quality=
("Content_Completeness","mean"),


satisfaction=
("Customer_Satisfaction","mean")

)

.sort_values(

"content_quality"

)

)


category_monitoring.head(10)

,products,content_quality,satisfaction
Category1,,,
بهداشت دهان و دندان,416,0.999273,0.762701
ضد تعریق,186,0.999715,0.780775
شامپو و مراقبت مو,790,0.999940,0.751817
بهداشت و مراقبت بدن,471,0.999949,0.786035
مراقبت پوست,1284,0.999966,0.761714
موسیقی بی‌کلام,28,1.000000,0.785000
موسیقی با کلام,42,1.000000,0.791127
مستند,7,1.000000,0.775000
مراقبت کودکان,1,1.000000,0.720000


In [7]:
content_issues = data[

    data["Content_Completeness"]

    <
    
    0.6

]


content_issues[

[
"title_fa",
"Category1",
"Brand",
"Content_Completeness"
]

].head(20)

,title_fa,Category1,Brand,Content_Completeness


In [8]:
def generate_action(row):

    actions=[]


    if row["Content_Completeness"] < 0.6:

        actions.append(
            "Improve product information"
        )


    if row["Customer_Satisfaction"] < 0.6:

        actions.append(
            "Analyze customer complaints"
        )


    if row["Engagement_Score"] < 0.2:

        actions.append(
            "Improve product presentation"
        )


    return ", ".join(actions)

In [9]:
content_issues["Recommended_Action"] = (

content_issues.apply(

generate_action,

axis=1

)

)


content_issues.head()

,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,...,Seller,Is_Fake,min_price_last_month,sub_category,Content_Completeness,Recommendation_Score,Customer_Satisfaction,Engagement_Score,Opportunity_Score,Recommended_Action


In [10]:
summary = {

"Number of products requiring attention":
len(content_issues),


"Average content quality":
round(
data["Content_Completeness"].mean(),
2
),


"Average customer satisfaction":
round(
data["Customer_Satisfaction"].mean(),
2
)

}


pd.DataFrame(

summary.items(),

columns=[
"Business Metric",
"Value"
]

)

,Business Metric,Value
0,Number of products requiring attention,0.00
1,Average content quality,1.00
2,Average customer satisfaction,0.75


In [11]:
import os


os.makedirs(

"../reports",

exist_ok=True

)


content_issues.to_csv(

"../reports/content_improvement_report.csv",

index=False,

encoding="utf-8-sig"

)


kpi_report.to_csv(

"../reports/kpi_summary.csv",

index=False,

encoding="utf-8-sig"

)


print(
"Reports generated successfully"
)

Reports generated successfully


# Future Improvements


## Generative AI


Possible extensions:


- Automatic product description rewriting

- AI generated improvement suggestions

- Review summarization

- Content quality chatbot assistant



## Production Deployment


Possible deployment:


Data Pipeline

↓

ML Model

↓

Dashboard

↓

Content Team
